# Data Discovery Dataset SMIC

## Peta proyek (baca dulu sebelum sidang)

Workspace aktif ada di `python test/`. Setiap arsitektur **diisolasi** di foldernya sendiri agar checkpoint `.pth` dan notebook tidak tercampur.

```text
python test/
├── discovery/          # EDA saja — tidak ada training
│   ├── smic_data_discovery.ipynb
│   └── mmew_data_discovery.ipynb
├── stgcn/              # model graf Face Mesh (468 node)
│   ├── stgcn_mmew_extraction.ipynb
│   ├── stgcn_mmew_pretraining.ipynb
│   ├── stgcn_smic_extraction.ipynb
│   ├── stgcn_smic_training.ipynb   # LOSO + transfer MMEW
│   ├── stgcn_mmew_pretrained.pth
│   ├── stgcn_best_model.pth
│   └── stgcn_mmew_landmarks/       # 300 file .npy MMEW
├── r3d18/              # 3D-CNN R3D-18 pada klip video
│   ├── r3d18_smic_pipeline.ipynb
│   └── r3d18_best_model.pth
├── vit/                # ViT pada gambar Optical Flow
│   ├── vit_optical_flow_pipeline.ipynb
│   └── vit_best_model.pth
├── dataset/            # data mentah + hasil ekstraksi bersama
│   ├── SMIC_all_cropped/           # frame BMP ter-crop & registrasi
│   ├── MMEW/                       # JPG mikro (sekuens) + makro (still)
│   ├── stgcn_smic_landmarks/       # .npy graf SMIC
│   └── vit_optical_flow_frames/    # JPG flow untuk ImageFolder
└── requirements.txt
```

**Alur skripsi yang benar:** discovery → ekstraksi fitur → (opsional pre-train) → LOSO SMIC. Ketiga model dievaluasi dengan protokol **LOSO** yang sama agar perbandingan adil (tanpa Subject Leakage dari split 80/20).

Notebook ini adalah **pintu masuk skripsi**: memetakan unit sampel SMIC sebelum model apa pun dilatih.

**Letak di peta:** `discovery/` → hanya membaca `../dataset/SMIC_all_cropped/`. Outputnya DataFrame indeks, bukan `.pth`.

Notebook ini berfungsi sebagai **mata-mata dataset**: menelusuri struktur folder, mengidentifikasi unit sampel, mengekstrak ID subjek dan label emosi, lalu menyusun indeks data dalam bentuk `pandas.DataFrame`.

## Temuan awal pada dataset lokal

Folder `SMIC_all_cropped` yang tersedia di workspace **tidak berisi video `.avi`**. Setiap klip disimpan sebagai sebuah folder yang berisi urutan frame `.bmp` hasil crop dan registrasi wajah.

Contoh satu sampel:

```text
SMIC_all_cropped/
└── HS/
    └── s1/
        └── micro/
            └── negative/
                └── s1_ne_01/
                    ├── reg_00001.bmp
                    ├── reg_00002.bmp
                    └── ...
```

Dengan demikian:

- `HS` / `NIR` / `VIS` adalah **modalitas perekaman**;
- `s1` adalah **ID subjek**;
- `micro` menandai sampel mikro-ekspresi;
- `negative`, `positive`, dan `surprise` adalah kelas emosi;
- `s1_ne_01` adalah **ID satu klip/sequence**;
- seluruh file `.bmp` di dalam folder klip merupakan frame-frame temporal dari satu kejadian.

Notebook tetap menyediakan scanner `.avi` sesuai kebutuhan awal. Jika tidak ditemukan AVI, scanner frame-sequence digunakan agar struktur dataset lokal tetap dapat dipetakan dengan benar.

> Notebook ini hanya melakukan **Data Discovery**. Tidak ada DataLoader, augmentasi, model, atau proses deep learning.


In [2]:
# =============================================================================
# 1. IMPORT LIBRARY DAN KONFIGURASI PATH DATASET
# =============================================================================
import os
import re
from pathlib import Path

import pandas as pd

# Notebook ini dijalankan dari folder discovery/, jadi path harus naik satu tingkat.
# Alasan: jika memakai Path('dataset/...') tanpa '../', Python mencari folder yang tidak ada.
DATASET_ROOT = Path('../dataset/SMIC_all_cropped').resolve()

# Kode di nama file SMIC singkatan (ne/po/sur). Model PyTorch butuh nama kelas yang sama
# dengan folder landmark/flow nanti, jadi dipetakan ke Negative/Positive/Surprise.
LABEL_MAP = {
    'ne': 'Negative',
    'po': 'Positive',
    'sur': 'Surprise',
}

# Ekstensi dicatat dalam lowercase agar pencarian tidak sensitif huruf besar/kecil.
VIDEO_EXTENSION = '.avi'
FRAME_EXTENSIONS = {'.bmp'}

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f'Folder SMIC_all_cropped tidak ditemukan: {DATASET_ROOT}\n'
        'Sesuaikan variabel DATASET_ROOT dengan lokasi dataset Anda.'
    )

print(f'Root dataset: {DATASET_ROOT}')
print(f'Isi level pertama: {sorted(p.name for p in DATASET_ROOT.iterdir())}')


Root dataset: /Users/stefanieagahari/Micro-Expression-Detector/python test/dataset/SMIC_all_cropped
Isi level pertama: ['.DS_Store', 'HS', 'NIR', 'VIS', 'li2013microexpressions.pdf']


## 2. Fungsi Parsing Identitas Sampel

Nama klip mikro-ekspresi mengikuti pola umum:

```text
<subject>_<emotion-code>_<sequence-number>
```

Contoh:

```text
s1_ne_01
│  │   └── sequence ke-01
│  └────── kode ne = Negative
└───────── subject s1
```

Fungsi berikut dibuat cukup fleksibel untuk menerima path file AVI maupun path folder frame-sequence. Informasi modalitas (`HS`, `NIR`, atau `VIS`) diambil dari posisi folder relatif terhadap `SMIC_all_cropped`.
**Mengapa parser ini penting?** Protokol LOSO membutuhkan `subject_id` yang konsisten. Jika ID subjek salah diekstrak, klip orang yang sama bisa bocor ke train dan val.


In [3]:
# =============================================================================
# 2. FUNGSI UNTUK MENGEKSTRAK SUBJECT, LABEL, DAN MODALITAS
# =============================================================================
def parse_sample_identity(sample_path: str | Path, dataset_root: Path = DATASET_ROOT) -> dict:
    """
    Mengekstrak metadata utama dari nama file atau folder sampel SMIC.

    Parameters
    ----------
    sample_path : str | Path
        Path file AVI atau folder klip berisi frame BMP.
    dataset_root : Path
        Root SMIC_all_cropped untuk menentukan modalitas.

    Returns
    -------
    dict
        Metadata subject_id, emotion_code, emotion_label, modality,
        sample_type, dan sequence_id.
    """
    path = Path(sample_path)

    # Untuk path ber-ekstensi gunakan stem; untuk frame-sequence gunakan nama folder klip.
    # Pemeriksaan suffix juga bekerja untuk contoh path yang filenya belum benar-benar ada.
    # File .avi punya ekstensi → pakai stem. Folder sequence tidak punya ekstensi → pakai name.
    # Alasan: satu fungsi harus menangani dua format penyimpanan SMIC (video vs folder BMP).
    sequence_id = path.stem if path.suffix else path.name
    sequence_lower = sequence_id.lower()

    # Pola menangkap s1_ne_01, s01-po-02, dan variasi pemisah _ / -.
    # Regex menangkap s1_ne_01 maupun s01-po-02. Urutan 'sur' sebelum 'ne' penting:
    # jika 'ne' dicek dulu, substring di dalam kata lain bisa salah cocok (aman di sini karena kode terpisah).
    pattern = re.compile(
        r'(?P<subject>s\d+)[_-](?P<emotion>sur|ne|po)(?:[_-]|$)',
        flags=re.IGNORECASE,
    )
    match = pattern.search(sequence_lower)

    subject_id = match.group('subject').lower() if match else None
    emotion_code = match.group('emotion').lower() if match else None
    emotion_label = LABEL_MAP.get(emotion_code, 'Unknown')

    # Fallback subject: cari komponen folder dengan pola s + angka.
    if subject_id is None:
        for part in path.parts:
            if re.fullmatch(r's\d+', part.lower()):
                subject_id = part.lower()
                break

    # Tentukan modalitas dari path relatif: HS, NIR, atau VIS.
    try:
        relative_parts = path.resolve().relative_to(dataset_root.resolve()).parts
    except ValueError:
        relative_parts = path.parts

    modality = next(
        (part.upper() for part in relative_parts if part.upper() in {'HS', 'NIR', 'VIS'}),
        'Unknown',
    )

    # Folder non_micro merupakan rekaman tanpa kejadian mikro-ekspresi.
    lower_parts = {part.lower() for part in path.parts}
    sample_type = 'Non-micro' if 'non_micro' in lower_parts else 'Micro'
    if sample_type == 'Non-micro':
        emotion_label = 'Non-micro'

    return {
        'subject_id': subject_id,
        'emotion_code': emotion_code,
        'emotion_label': emotion_label,
        'modality': modality,
        'sample_type': sample_type,
        'sequence_id': sequence_id,
    }


# Sanity check parser menggunakan contoh nama yang diberikan.
example_path = DATASET_ROOT / 'HS' / 's1' / 'micro' / 'negative' / 's1_ne_01' / 's1_ne_01.avi'
print(parse_sample_identity(example_path))


{'subject_id': 's1', 'emotion_code': 'ne', 'emotion_label': 'Negative', 'modality': 'HS', 'sample_type': 'Micro', 'sequence_id': 's1_ne_01'}


## 3. Scanner File `.avi`

Cell berikut memenuhi kebutuhan awal: menelusuri seluruh subfolder menggunakan `os.walk`, mencari file `.avi`, mengekstrak metadata, dan membentuk DataFrame.

Jika DataFrame kosong, hal tersebut bukan error pada kode—artinya distribusi SMIC lokal memang tidak menggunakan container video AVI.

In [4]:
# =============================================================================
# 3. TELUSURI SELURUH FOLDER DAN CARI FILE VIDEO .AVI
# =============================================================================
def discover_avi_files(dataset_root: str | Path) -> pd.DataFrame:
    """Mencari seluruh AVI secara rekursif dan menyusun metadata ke DataFrame."""
    dataset_root = Path(dataset_root).resolve()
    records = []

    # os.walk menelusuri root, seluruh subfolder, dan nama file di setiap level.
    for current_root, _, filenames in os.walk(dataset_root):
        for filename in filenames:
            if Path(filename).suffix.lower() != VIDEO_EXTENSION:
                continue

            video_path = (Path(current_root) / filename).resolve()
            metadata = parse_sample_identity(video_path, dataset_root)

            records.append({
                'video_path': str(video_path),
                **metadata,
                'data_format': 'AVI video',
            })

    columns = [
        'video_path', 'subject_id', 'emotion_code', 'emotion_label',
        'modality', 'sample_type', 'sequence_id', 'data_format',
    ]
    return pd.DataFrame(records, columns=columns)


avi_df = discover_avi_files(DATASET_ROOT)

print('Lima baris pertama hasil pencarian AVI:')
display(avi_df.head(5))
print(f'\nTotal file AVI ditemukan: {len(avi_df)}')

if avi_df.empty:
    print(
        '\nTidak ditemukan file .avi pada SMIC_all_cropped lokal. '
        'Lanjutkan ke scanner frame-sequence BMP pada cell berikutnya.'
    )
else:
    print('\nTotal video per kelas emosi:')
    print(avi_df['emotion_label'].value_counts(dropna=False))

Lima baris pertama hasil pencarian AVI:


,video_path,subject_id,emotion_code,emotion_label,modality,sample_type,sequence_id,data_format



Total file AVI ditemukan: 0

Tidak ditemukan file .avi pada SMIC_all_cropped lokal. Lanjutkan ke scanner frame-sequence BMP pada cell berikutnya.


## 4. Scanner Frame-Sequence `.bmp` (Format Aktual Dataset Lokal)

Untuk versi dataset ini, **satu folder klip diperlakukan sebagai satu sampel video**. Scanner tidak membuat satu baris per frame karena hal itu akan mengubah unit analisis dan menyebabkan label klip terduplikasi ribuan kali.

Setiap baris DataFrame mewakili satu sequence dan mencatat:

- path lengkap folder klip,
- subject ID,
- label emosi,
- modalitas,
- tipe sampel (`Micro` / `Non-micro`),
- jumlah frame,
- path frame pertama dan terakhir, serta
- format penyimpanan data.
**Inti metodologi:** 1 folder klip = 1 kejadian mikro-ekspresi. Jangan pernah membuat 1 baris per BMP, karena itu menggandakan label dan menghancurkan informasi temporal untuk 3D-CNN/ST-GCN.


In [5]:
# =============================================================================
# 4. TEMUKAN SETIAP FOLDER KLIP YANG BERISI URUTAN FRAME .BMP
# =============================================================================
def frame_sort_key(frame_path: Path):
    """Mengurutkan frame berdasarkan angka terakhir pada nama file, bukan alfabet biasa."""
    numbers = re.findall(r'\d+', frame_path.stem)
    # Urut numerik, bukan alfabet. Tanpa ini, reg_10.bmp muncul sebelum reg_2.bmp
    # dan gerak temporal wajah jadi kacau sebelum masuk model.
    return (0, int(numbers[-1])) if numbers else (1, frame_path.name)


def discover_frame_sequences(dataset_root: str | Path) -> pd.DataFrame:
    """Menyusun satu record untuk setiap folder klip yang memiliki frame BMP."""
    dataset_root = Path(dataset_root).resolve()
    records = []

    for current_root, _, filenames in os.walk(dataset_root):
        frame_names = [
            name for name in filenames
            if Path(name).suffix.lower() in FRAME_EXTENSIONS
        ]

        # Folder tanpa frame bukan unit sampel sehingga dilewati.
        if not frame_names:
            continue

        sequence_path = Path(current_root).resolve()
        frame_paths = sorted(
            (sequence_path / name for name in frame_names),
            key=frame_sort_key,
        )
        metadata = parse_sample_identity(sequence_path, dataset_root)

        records.append({
            'sequence_path': str(sequence_path),
            **metadata,
            'num_frames': len(frame_paths),
            'first_frame': str(frame_paths[0]),
            'last_frame': str(frame_paths[-1]),
            'data_format': 'BMP frame sequence',
        })

    columns = [
        'sequence_path', 'subject_id', 'emotion_code', 'emotion_label',
        'modality', 'sample_type', 'sequence_id', 'num_frames',
        'first_frame', 'last_frame', 'data_format',
    ]
    return pd.DataFrame(records, columns=columns)


sequence_df = discover_frame_sequences(DATASET_ROOT)

print('Lima baris pertama indeks sequence SMIC:')
display(sequence_df.head(5))
print(f'\nTotal seluruh sequence (micro + non-micro): {len(sequence_df)}')
print(f'Total subjek unik: {sequence_df["subject_id"].nunique()}')
print(f'Modalitas ditemukan: {sorted(sequence_df["modality"].unique())}')


Lima baris pertama indeks sequence SMIC:


,sequence_path,subject_id,emotion_code,emotion_label,modality,sample_type,sequence_id,num_frames,first_frame,last_frame,data_format
0,/Users/stefanieagahari/Micro-Expression-Detect...,s19,po,Positive,VIS,Micro,s19_po_01,9,/Users/stefanieagahari/Micro-Expression-Detect...,/Users/stefanieagahari/Micro-Expression-Detect...,BMP frame sequence
1,/Users/stefanieagahari/Micro-Expression-Detect...,s19,sur,Surprise,VIS,Micro,s19_sur_01,13,/Users/stefanieagahari/Micro-Expression-Detect...,/Users/stefanieagahari/Micro-Expression-Detect...,BMP frame sequence
2,/Users/stefanieagahari/Micro-Expression-Detect...,s19,None,Non-micro,VIS,Non-micro,s19_n2,12,/Users/stefanieagahari/Micro-Expression-Detect...,/Users/stefanieagahari/Micro-Expression-Detect...,BMP frame sequence
3,/Users/stefanieagahari/Micro-Expression-Detect...,s19,None,Non-micro,VIS,Non-micro,s19_n1,9,/Users/stefanieagahari/Micro-Expression-Detect...,/Users/stefanieagahari/Micro-Expression-Detect...,BMP frame sequence
4,/Users/stefanieagahari/Micro-Expression-Detect...,s11,po,Positive,VIS,Micro,s11_po_01,12,/Users/stefanieagahari/Micro-Expression-Detect...,/Users/stefanieagahari/Micro-Expression-Detect...,BMP frame sequence



Total seluruh sequence (micro + non-micro): 612
Total subjek unik: 16
Modalitas ditemukan: ['HS', 'NIR', 'VIS']


## 5. Distribusi Kelas dan Pemeriksaan Keseimbangan Data

Analisis keseimbangan kelas harus dilakukan pada **jumlah klip**, bukan jumlah frame. Klip panjang memiliki frame lebih banyak, tetapi tetap merepresentasikan satu kejadian mikro-ekspresi.

Sampel `non_micro` dipisahkan dari tiga kelas emosi karena rekaman tersebut adalah kontrol/segmen tanpa kejadian mikro-ekspresi. Untuk tugas klasifikasi tiga kelas, gunakan hanya baris dengan `sample_type == 'Micro'`.
**Imbalance SMIC HS:** Negative 70, Positive 51, Surprise 43. Itulah alasan skripsi memakai **Macro F1** dan **UAR**, bukan accuracy semata. Non-micro (164 klip) **tidak** masuk eksperimen klasifikasi 3 kelas.


In [6]:
# =============================================================================
# 5. RINGKASAN JUMLAH KLIP PER KELAS DAN MODALITAS
# =============================================================================
# Ambil hanya klip mikro-ekspresi dengan label yang berhasil dikenali.
# Filter ketat: hanya Micro + 3 kelas. Non-micro dan label Unknown dibuang agar tidak masuk LOSO.
micro_df = sequence_df[
    (sequence_df['sample_type'] == 'Micro')
    & (sequence_df['emotion_label'].isin(LABEL_MAP.values()))
].copy()

print('Lima baris pertama DataFrame mikro-ekspresi:')
display(micro_df.head(5))

print('\nTotal klip mikro-ekspresi per kelas (semua modalitas):')
class_counts = micro_df['emotion_label'].value_counts().reindex(
    ['Negative', 'Positive', 'Surprise'],
    fill_value=0,
)
print(class_counts)

print('\nTotal klip per kelas dan modalitas:')
class_by_modality = pd.crosstab(
    micro_df['modality'],
    micro_df['emotion_label'],
).reindex(
    index=['HS', 'NIR', 'VIS'],
    columns=['Negative', 'Positive', 'Surprise'],
    fill_value=0,
)
display(class_by_modality)

print('\nTotal sequence non-micro per modalitas:')
print(
    sequence_df[sequence_df['sample_type'] == 'Non-micro']['modality']
    .value_counts()
    .reindex(['HS', 'NIR', 'VIS'], fill_value=0)
)

# Rasio kelas terbesar terhadap terkecil sebagai indikator sederhana imbalance.
nonzero_counts = class_counts[class_counts > 0]
if len(nonzero_counts) > 1:
    imbalance_ratio = nonzero_counts.max() / nonzero_counts.min()
    print(f'\nRasio kelas terbesar : terkecil = {imbalance_ratio:.2f} : 1')
    print('Kesimpulan: data tidak sepenuhnya balanced.' if imbalance_ratio > 1.2 else 'Kesimpulan: distribusi relatif balanced.')


Lima baris pertama DataFrame mikro-ekspresi:


,sequence_path,subject_id,emotion_code,emotion_label,modality,sample_type,sequence_id,num_frames,first_frame,last_frame,data_format
0,/Users/stefanieagahari/Micro-Expression-Detect...,s19,po,Positive,VIS,Micro,s19_po_01,9,/Users/stefanieagahari/Micro-Expression-Detect...,/Users/stefanieagahari/Micro-Expression-Detect...,BMP frame sequence
1,/Users/stefanieagahari/Micro-Expression-Detect...,s19,sur,Surprise,VIS,Micro,s19_sur_01,13,/Users/stefanieagahari/Micro-Expression-Detect...,/Users/stefanieagahari/Micro-Expression-Detect...,BMP frame sequence
4,/Users/stefanieagahari/Micro-Expression-Detect...,s11,po,Positive,VIS,Micro,s11_po_01,12,/Users/stefanieagahari/Micro-Expression-Detect...,/Users/stefanieagahari/Micro-Expression-Detect...,BMP frame sequence
5,/Users/stefanieagahari/Micro-Expression-Detect...,s11,po,Positive,VIS,Micro,s11_po_03,11,/Users/stefanieagahari/Micro-Expression-Detect...,/Users/stefanieagahari/Micro-Expression-Detect...,BMP frame sequence
6,/Users/stefanieagahari/Micro-Expression-Detect...,s11,po,Positive,VIS,Micro,s11_po_02,12,/Users/stefanieagahari/Micro-Expression-Detect...,/Users/stefanieagahari/Micro-Expression-Detect...,BMP frame sequence



Total klip mikro-ekspresi per kelas (semua modalitas):
emotion_label
Negative    116
Positive    107
Surprise     83
Name: count, dtype: int64

Total klip per kelas dan modalitas:


emotion_label,Negative,Positive,Surprise
modality,,,
HS,70,51,43
NIR,23,28,20
VIS,23,28,20



Total sequence non-micro per modalitas:
modality
HS     164
NIR     71
VIS     71
Name: count, dtype: int64

Rasio kelas terbesar : terkecil = 1.40 : 1
Kesimpulan: data tidak sepenuhnya balanced.


## 6. Cara Memahami Isi Dataset dan Kebutuhan Tahap Berikutnya

### A. Isi utama dataset lokal

Dataset memiliki tiga modalitas perekaman:

- **HS (High Speed):** sequence berkecepatan tinggi dan merupakan modalitas utama untuk menangkap perubahan wajah yang sangat singkat;
- **NIR (Near Infrared):** sequence inframerah dekat, lebih tahan terhadap perubahan pencahayaan;
- **VIS (Visible):** sequence spektrum cahaya tampak.

Setiap modalitas memiliki subjek dan sequence tersendiri. Pada data lokal saat notebook dibuat:

| Modalitas | Micro Negative | Micro Positive | Micro Surprise | Non-micro | Total sequence |
|---|---:|---:|---:|---:|---:|
| HS | 70 | 51 | 43 | 164 | 328 |
| NIR | 23 | 28 | 20 | 71 | 142 |
| VIS | 23 | 28 | 20 | 71 | 142 |

Distribusi tiga kelas mikro-ekspresi tidak sepenuhnya seimbang. Selain itu, NIR dan VIS dapat merekam kejadian yang berpasangan; keduanya tidak boleh langsung dianggap sebagai sampel independen tanpa memeriksa protokol dataset.

### B. Arti kode nama klip

- `ne` → **Negative**
- `po` → **Positive**
- `sur` → **Surprise**
- folder `non_micro` → segmen tanpa kejadian mikro-ekspresi

Label emosi dapat diambil dari folder kelas (`negative`, `positive`, `surprise`) maupun kode pada ID klip. Menyimpan keduanya dalam DataFrame berguna untuk validasi silang jika terdapat nama yang tidak konsisten.

### C. Unit sampel yang benar

Unit sampel untuk model spatiotemporal adalah **satu folder sequence**, bukan satu file BMP. Frame-frame di dalam sequence harus dipertahankan urutannya. Jika setiap frame dianggap sebagai sampel independen, informasi temporal akan hilang dan evaluasi dapat menjadi bias.

### D. Kebutuhan sebelum membuat DataLoader

1. **Tentukan modalitas eksperimen.** Mulai dari HS saja lebih sederhana dan menghindari pencampuran domain HS/NIR/VIS.
2. **Pisahkan berdasarkan subjek.** Frame atau klip dari subjek yang sama tidak boleh tersebar ke train dan test karena menyebabkan subject leakage.
3. **Tentukan protokol evaluasi.** Untuk mikro-ekspresi, LOSO (*Leave-One-Subject-Out*) lebih representatif dibanding random split biasa.
4. **Pertahankan urutan temporal.** Urutkan frame berdasarkan nomor frame dan catat panjang sequence.
5. **Tentukan panjang input temporal.** Sequence memiliki jumlah frame berbeda; nantinya diperlukan sampling, padding, atau interpolasi temporal.
6. **Atasi class imbalance.** Pertimbangkan class-weighted loss, balanced sampler, serta metrik macro-F1 dan UAR.
7. **Hindari duplikasi modalitas.** Jika NIR dan VIS merekam kejadian yang sama, pengelompokan harus dilakukan berdasarkan event/subjek saat split.
8. **Validasi kualitas data.** Periksa folder kosong, frame korup, label `Unknown`, serta konsistensi resolusi.

### E. DataFrame yang dipakai selanjutnya

Untuk eksperimen tiga kelas pada HS, subset awal yang aman adalah:

```python
hs_micro_df = micro_df[micro_df['modality'] == 'HS'].reset_index(drop=True)
```

DataFrame tersebut sudah memiliki path sequence, subject, label, jumlah frame, serta frame awal/akhir. Itu merupakan fondasi yang tepat untuk tahap selanjutnya, tetapi notebook ini sengaja berhenti pada Data Discovery sesuai ruang lingkup eksplorasi.